# 기프트코 상품 추천 RAG v2 (profile 단위 집계)
## 거래데이터를 구매처/상품분류/시기 단위로 묶어서 검색

v1은 거래 1건(row)을 그대로 임베딩해서 검색했습니다. v2는 거래데이터를
**구매처분류 + 상품분류 + 주문월 + 행사/대상/시즌** 기준으로 묶은 "거래 Profile" 단위로 집계한 뒤,
그 profile을 임베딩해서 검색합니다.

상품데이터 쪽은 v1과 동일하게 **상품 row 단위 임베딩**을 그대로 씁니다. 바뀌는 건 거래데이터 쪽뿐입니다.

## v1과 달라진 점

| 구분 | v1 (row 단위) | v2 (profile 단위) |
|---|---|---|
| 거래데이터 단위 | 거래 56,599건 각각을 문서로 취급 | 구매처×상품분류×주문월×행사/대상/시즌으로 묶어 문서 수를 크게 줄임 |
| 거래 임베딩 개수 | 거래 row 수만큼 | profile 수만큼 (row보다 훨씬 적음) |
| 검색 점수 | 의미유사도 55% + 리드타임 25% + 조건 20% | 의미유사도 50% + 리드타임 20% + 조건 15% + **거래량 점수 15%** |
| 거래량 반영 | 없음 (거래 1건은 그냥 1건) | 있음 — 거래가 많이 쌓인 profile을 조금 더 신뢰 |
| 대표 상품/구매처 | 필요 없음 (row 자체가 구체적) | profile 안에 여러 거래가 섞여 있어 "대표 샘플"로 보관 |

## 이 노트북에서 보는 RAG 4단계

이 부분은 v1과 동일합니다 — 바뀌는 건 Retrieval 단계에서 "무엇을 문서로 볼 것인가"입니다.

| 단계 | RAG에서의 역할 | v2에서 하는 일 |
|---|---|---|
| **1. Indexing** | 검색 대상 문서를 임베딩으로 변환 | 상품은 row 단위, **거래는 profile 단위**로 문서를 만들고 임베딩 |
| **2. Query 이해** | 질문에서 검색 조건 추출 | v1과 동일 (LLM 조건 추출 + 리드타임 계산) |
| **3. Retrieval** | 관련 문서 찾기 | 유사 거래 **profile** 검색 → profile 신호 추출 → 상품 후보 랭킹 |
| **4. Generation** | 근거 기반 답변 생성 | v1과 동일 |

## 0. 설치

v1과 동일합니다. 이미 v1을 실행해보셨다면 건너뛰어도 됩니다.

```bash
python -m pip install pandas openpyxl scikit-learn numpy ollama
ollama pull bge-m3
ollama pull gemma3:4b
```

In [ ]:
# 필요 시 한 번만 실행
# !pip install pandas openpyxl scikit-learn numpy ollama

## 1. 기본 설정 및 파일 경로

*이 노트북이 독립적으로 실행되도록, v1의 공통 준비과정(설정~참고월 계산)을 그대로 복사했습니다.*

In [ ]:
from pathlib import Path
import re
import html
import json
import hashlib
import warnings

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 180)

PRODUCT_PATH = Path("data/상품데이터_샘플100.xlsx")
TRADE_PATH = Path("data/거래데이터_샘플100.xlsx")

OUTPUT_DIR = Path("output")
CACHE_DIR = Path("cache")
OUTPUT_DIR.mkdir(exist_ok=True)
CACHE_DIR.mkdir(exist_ok=True)

USE_OLLAMA_EMBEDDING = True
EMBED_MODEL = "bge-m3"
LLM_MODEL = "gemma3:4b"

DEFAULT_TOP_K = 5

print("PRODUCT_PATH:", PRODUCT_PATH)
print("TRADE_PATH:", TRADE_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())

## 2. 파일 로딩 및 컬럼 확인

In [ ]:
product_raw = pd.read_excel(PRODUCT_PATH, dtype=str).fillna("")
trade_raw = pd.read_excel(TRADE_PATH, dtype=str).fillna("")

print("상품데이터 shape:", product_raw.shape)
print("거래데이터 shape:", trade_raw.shape)
display(product_raw.head(3))
display(trade_raw.head(3))

## 3. 공통 전처리 함수

In [ ]:
def clean_text(x):
    """HTML 태그/엔티티를 제거하고 공백을 정리합니다."""
    if pd.isna(x):
        return ""
    x = str(x)
    x = re.sub(r"<[^>]+>", " ", x)
    x = html.unescape(x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


def to_number(x):
    """'3,000원' 같은 문자열에서 숫자만 뽑아 float로 변환합니다."""
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s in ["", "-", "nan", "None", "null"]:
        return np.nan
    s = re.sub(r"[^0-9.]", "", s)
    if s == "":
        return np.nan
    try:
        return float(s)
    except Exception:
        return np.nan


def safe_col(df, col):
    """컬럼이 없으면 빈 문자열 시리즈를 돌려줘서 KeyError를 막습니다."""
    if col in df.columns:
        return df[col].fillna("").astype(str)
    return pd.Series([""] * len(df), index=df.index)


def combine_text(*parts):
    return clean_text(" ".join([str(p) for p in parts if str(p).strip()]))


def join_top_values(series, n=8):
    """profile 집계 시 그룹 안의 값들 중 중복 제거 후 상위 n개만 '/'로 이어 붙입니다."""
    values = [clean_text(x) for x in series.tolist() if clean_text(x)]
    values = list(dict.fromkeys(values))
    return " / ".join(values[:n])


def normalize_vector_matrix(mat: np.ndarray) -> np.ndarray:
    """임베딩 행렬을 행 단위 L2 정규화합니다 (코사인 유사도를 내적으로 계산하기 위함)."""
    mat = np.array(mat, dtype=np.float32)
    norms = np.linalg.norm(mat, axis=1, keepdims=True)
    norms[norms == 0] = 1
    return mat / norms


def cosine_scores(query_vec: np.ndarray, matrix: np.ndarray) -> np.ndarray:
    """정규화된 query 벡터와 문서 행렬 간 코사인 유사도(=내적)를 계산합니다."""
    query_vec = np.array(query_vec, dtype=np.float32)
    if query_vec.ndim == 1:
        query_vec = query_vec.reshape(1, -1)
    query_norm = np.linalg.norm(query_vec, axis=1, keepdims=True)
    query_norm[query_norm == 0] = 1
    query_vec = query_vec / query_norm
    return (matrix @ query_vec.T).ravel()


def text_hash(texts):
    """임베딩 캐시 파일명을 만들기 위한 내용 해시입니다."""
    joined = "\n".join([str(x) for x in texts])
    return hashlib.md5(joined.encode("utf-8")).hexdigest()[:10]

## 4. 상품데이터 정규화

*v1과 동일 — 상품은 v2에서도 row 단위 임베딩을 그대로 씁니다.*

In [ ]:
product_df = product_raw.copy()

product_df["product_id"] = safe_col(product_df, "상품번호")
product_df.loc[product_df["product_id"].str.strip() == "", "product_id"] = safe_col(product_df, "상품코드")

product_df["brand"] = safe_col(product_df, "브랜드").map(clean_text)
product_df["product_name"] = safe_col(product_df, "상품명").map(clean_text)
product_df["model_name"] = safe_col(product_df, "모델명").map(clean_text)

product_df["price"] = safe_col(product_df, "상품판매가").map(to_number)
product_df["moq"] = safe_col(product_df, "최소구매수량").map(to_number)

product_df["category_large"] = safe_col(product_df, "대 카테고리").map(clean_text)
product_df["category_middle"] = safe_col(product_df, "중 카테고리").map(clean_text)
product_df["category_small"] = safe_col(product_df, "소 카테고리").map(clean_text)
product_df["category_detail"] = safe_col(product_df, "세분류").map(clean_text)

product_df["category_path"] = (
    product_df["category_large"] + " > " +
    product_df["category_middle"] + " > " +
    product_df["category_small"] + " > " +
    product_df["category_detail"]
).map(clean_text)

product_df["keywords"] = safe_col(product_df, "검색키워드").map(clean_text)
product_df["summary"] = safe_col(product_df, "간략한설명").map(clean_text)
product_df["detail_text"] = safe_col(product_df, "상품내용").map(clean_text)
product_df["manufacturer"] = safe_col(product_df, "제조사").map(clean_text)
product_df["origin"] = safe_col(product_df, "원산지").map(clean_text)
product_df["image"] = safe_col(product_df, "큰이미지").map(clean_text)

product_df["product_search_text"] = (
    "[상품명] " + product_df["product_name"] + "\n" +
    "[브랜드] " + product_df["brand"] + "\n" +
    "[모델명] " + product_df["model_name"] + "\n" +
    "[카테고리] " + product_df["category_path"] + "\n" +
    "[검색키워드] " + product_df["keywords"] + "\n" +
    "[설명] " + product_df["summary"] + " " + product_df["detail_text"] + "\n" +
    "[제조/원산지] " + product_df["manufacturer"] + " " + product_df["origin"]
).map(clean_text)

product_df = product_df[product_df["product_name"].str.len() > 0].reset_index(drop=True)

print("정규화 상품 수:", len(product_df))
display(product_df[["product_id", "product_name", "price", "moq", "category_path", "keywords"]].head(10))

## 5. 거래데이터 정규화 (row 단위)

*이 단계는 v1과 동일합니다. profile 집계는 이 row 데이터를 바탕으로 다음 섹션에서 만듭니다.*

In [ ]:
trade_df = trade_raw.copy()

buyer_name_col = "구매처 명 "
if buyer_name_col not in trade_df.columns and "구매처 명" in trade_df.columns:
    buyer_name_col = "구매처 명"

trade_df["buyer_type_large"] = safe_col(trade_df, "구매처 분류(대)").map(clean_text)
trade_df["buyer_type_middle"] = safe_col(trade_df, "구매처 분류(중)").map(clean_text)
trade_df["buyer_type_small"] = safe_col(trade_df, "구매처 분류(소)").map(clean_text)
trade_df["buyer_type_detail"] = safe_col(trade_df, "구매처 분류(세)").map(clean_text)
trade_df["buyer_name"] = safe_col(trade_df, buyer_name_col).map(clean_text)

trade_df["buyer_type_path"] = (
    trade_df["buyer_type_large"] + " > " +
    trade_df["buyer_type_middle"] + " > " +
    trade_df["buyer_type_small"] + " > " +
    trade_df["buyer_type_detail"]
).map(clean_text)

trade_df["date_raw"] = safe_col(trade_df, "날짜").map(clean_text)
trade_df["order_date"] = pd.to_datetime(trade_df["date_raw"], errors="coerce")
trade_df["order_month"] = trade_df["order_date"].dt.month

trade_df["trade_product_name"] = safe_col(trade_df, "상품").map(clean_text)

trade_df["trade_category_large"] = safe_col(trade_df, "상품분류(대)").map(clean_text)
trade_df["trade_category_middle"] = safe_col(trade_df, "상품분류(중)").map(clean_text)
trade_df["trade_category_small"] = safe_col(trade_df, "상품분류(소)").map(clean_text)

trade_df["trade_category_path"] = (
    trade_df["trade_category_large"] + " > " +
    trade_df["trade_category_middle"] + " > " +
    trade_df["trade_category_small"]
).map(clean_text)

trade_df["bulk_price"] = safe_col(trade_df, "대량가격(원)").map(to_number)
trade_df["middle_price"] = safe_col(trade_df, "중간가격(원)").map(to_number)
trade_df["small_price"] = safe_col(trade_df, "소량가격(원)").map(to_number)
trade_df["trade_moq"] = safe_col(trade_df, "최소구매수량").map(to_number)

trade_df["event_filter"] = safe_col(trade_df, "행사별(필터)").map(clean_text)
trade_df["target_filter"] = safe_col(trade_df, "대상별(필터)").map(clean_text)
trade_df["season_filter"] = safe_col(trade_df, "시즌별(필터)").map(clean_text)
trade_df["print_method"] = safe_col(trade_df, "인쇄 방법").map(clean_text)
trade_df["memo"] = safe_col(trade_df, "비  고").map(clean_text)

trade_df = trade_df[trade_df["trade_product_name"].str.len() > 0].reset_index(drop=True)

print("정규화 거래 수 (row 단위):", len(trade_df))
display(trade_df[[
    "date_raw", "order_month", "buyer_type_path", "buyer_name",
    "trade_product_name", "trade_category_path",
    "bulk_price", "middle_price", "small_price", "trade_moq",
    "event_filter", "target_filter", "season_filter",
]].head(10))

## 6. 거래 Profile 생성 (v2 핵심)

*RAG 단계: Indexing — 문서 단위 자체를 바꾸는 부분입니다. 거래 row를 그대로 쓰지 않고,
아래 기준으로 묶어서 "거래 Profile"이라는 새로운 문서를 만듭니다.*

```text
묶는 기준 = 구매처분류 + 상품분류 + 주문월 + 행사필터 + 대상필터 + 시즌필터
```

`구매처명`과 `거래 상품명`은 묶는 기준에 넣지 않고, 대표 샘플로만 보관합니다 (안 그러면 profile 수가 row 수와 비슷해져 집계 의미가 없어집니다).

In [ ]:
def make_trade_profiles(trade_df: pd.DataFrame) -> pd.DataFrame:
    """거래 row를 구매처분류/상품분류/주문월/행사·대상·시즌 기준으로 묶어 profile 단위로 집계합니다."""
    group_cols = [
        "buyer_type_large", "buyer_type_middle", "buyer_type_small", "buyer_type_detail",
        "trade_category_large", "trade_category_middle", "trade_category_small",
        "order_month", "event_filter", "target_filter", "season_filter",
    ]
    group_cols = [c for c in group_cols if c in trade_df.columns]

    profile_df = (
        trade_df.groupby(group_cols, dropna=False)
        .agg(
            trade_count=("trade_product_name", "count"),
            sample_products=("trade_product_name", lambda x: join_top_values(x, n=10)),
            sample_buyer_names=("buyer_name", lambda x: join_top_values(x, n=8)),
            sample_print_methods=("print_method", lambda x: join_top_values(x, n=5)),
            median_bulk_price=("bulk_price", "median"),
            median_middle_price=("middle_price", "median"),
            median_small_price=("small_price", "median"),
            median_moq=("trade_moq", "median"),
        )
        .reset_index()
    )

    profile_df["buyer_type_path"] = (
        profile_df["buyer_type_large"].fillna("").astype(str) + " > " +
        profile_df["buyer_type_middle"].fillna("").astype(str) + " > " +
        profile_df["buyer_type_small"].fillna("").astype(str) + " > " +
        profile_df["buyer_type_detail"].fillna("").astype(str)
    ).map(clean_text)

    profile_df["trade_category_path"] = (
        profile_df["trade_category_large"].fillna("").astype(str) + " > " +
        profile_df["trade_category_middle"].fillna("").astype(str) + " > " +
        profile_df["trade_category_small"].fillna("").astype(str)
    ).map(clean_text)

    profile_df["trade_profile_id"] = [f"TP-{i+1:06d}" for i in range(len(profile_df))]

    # profile 1건 = 문서 1개 (임베딩 대상)
    profile_df["trade_profile_search_text"] = (
        "[구매처분류] " + profile_df["buyer_type_path"] + "\n" +
        "[주문월] " + profile_df["order_month"].fillna("").astype(str) + "월\n" +
        "[상품분류] " + profile_df["trade_category_path"] + "\n" +
        "[대표상품] " + profile_df["sample_products"].fillna("").astype(str) + "\n" +
        "[대표구매처] " + profile_df["sample_buyer_names"].fillna("").astype(str) + "\n" +
        "[행사/대상/시즌] " +
            profile_df["event_filter"].fillna("").astype(str) + " " +
            profile_df["target_filter"].fillna("").astype(str) + " " +
            profile_df["season_filter"].fillna("").astype(str) + "\n" +
        "[인쇄방법] " + profile_df["sample_print_methods"].fillna("").astype(str) + "\n" +
        "[거래수] " + profile_df["trade_count"].astype(str)
    ).map(clean_text)

    # 거래가 많이 쌓인 profile을 조금 더 신뢰하기 위한 점수 (log로 완만하게)
    profile_df["trade_count_weight"] = np.log1p(profile_df["trade_count"])
    max_w = profile_df["trade_count_weight"].max()
    profile_df["trade_count_score"] = profile_df["trade_count_weight"] / max_w if max_w and max_w > 0 else 0.0

    return profile_df


trade_profile_df = make_trade_profiles(trade_df)

print("원본 거래 row 수:", len(trade_df))
print("거래 profile 수:", len(trade_profile_df))
print("감소율:", f"{(1 - len(trade_profile_df) / max(len(trade_df), 1)) * 100:.1f}%")

display(trade_profile_df[[
    "trade_profile_id", "trade_count", "order_month", "buyer_type_path", "trade_category_path",
    "sample_products", "event_filter", "target_filter", "season_filter", "trade_count_score",
]].head(20))

## 7. 질문 조건 추출

*v1과 동일한 함수입니다. `profile`이라는 이름을 쓰지 않고 `query_condition`으로 부릅니다 —
이 노트북에서는 "profile"이 거래 Profile(위 6번 섹션)을 가리키므로, 질문 조건과 헷갈리지 않게 구분합니다.*

In [ ]:
SEASON_MONTHS = {
    "봄": [3, 4, 5],
    "여름": [6, 7, 8],
    "가을": [9, 10, 11],
    "겨울": [12, 1, 2],
}


def extract_basic_conditions(query: str) -> dict:
    """정규식만으로 예산/수량/월/계절을 추출하는 fallback입니다."""
    q = str(query)

    budget = None
    m = re.search(r"(\d+(?:,\d+)*)\s*원", q)
    if m:
        budget = int(m.group(1).replace(",", ""))
    if budget is None:
        m = re.search(r"(\d+)\s*천\s*원", q)
        if m:
            budget = int(m.group(1)) * 1000
    if budget is None:
        m = re.search(r"(\d+)\s*만\s*원", q)
        if m:
            budget = int(m.group(1)) * 10000

    quantity = None
    m = re.search(r"(\d+(?:,\d+)*)\s*(?:개|ea|EA|pcs|PCS)", q)
    if m:
        quantity = int(m.group(1).replace(",", ""))

    event_month = None
    m = re.search(r"(1[0-2]|[1-9])\s*월", q)
    if m:
        event_month = int(m.group(1))

    season = next((s for s in SEASON_MONTHS if s in q), None)

    return {
        "raw_query": q,
        "buyer_context": "",
        "event_context": "",
        "purpose": "",
        "season": season,
        "event_month": event_month,
        "budget_max": budget,
        "quantity": quantity,
        "style_keywords": [],
        "must_have": [],
        "extraction_method": "regex_fallback",
    }


def safe_json_loads(text: str) -> dict:
    """LLM 응답에서 코드블록/잡음을 제거하고 JSON 객체만 파싱합니다."""
    text = str(text).strip()
    text = re.sub(r"^```json", "", text).strip()
    text = re.sub(r"^```", "", text).strip()
    text = re.sub(r"```$", "", text).strip()
    start, end = text.find("{"), text.rfind("}")
    if start >= 0 and end > start:
        text = text[start:end + 1]
    return json.loads(text)


def extract_query_condition(query: str, model: str = LLM_MODEL) -> dict:
    """Ollama LLM으로 질문을 구조화합니다. 실패하면 정규식 fallback을 씁니다."""
    fallback = extract_basic_conditions(query)

    try:
        import ollama
    except ModuleNotFoundError:
        return fallback

    system_prompt = """
너는 판촉물 상품 추천을 위한 조건 추출기다.
사용자 질문에서 추천에 필요한 조건만 JSON으로 추출한다.

주의:
- 상품 추천 사전을 만들지 않는다.
- "여름이면 선풍기"처럼 상품군을 임의 확장하지 않는다.
- 사용자가 말한 구매처, 행사, 목적, 계절, 월, 예산, 수량, 스타일만 구조화한다.
- 모르는 값은 null 또는 빈 문자열/빈 리스트로 둔다.
- 반드시 JSON만 출력한다.
"""

    user_prompt = f"""
아래 사용자 요청을 JSON으로 구조화해줘.

[사용자 요청]
{query}

[출력 JSON 스키마]
{{
  "buyer_context": "구매처/업종/기관/대상 예: 병원, 대학교, 기업, 어린이집",
  "event_context": "행사/상황 예: 개원, 창립기념, 박람회, 여름행사",
  "purpose": "용도 예: 답례품, 사은품, 홍보물, 기념품",
  "season": "봄/여름/가을/겨울 중 하나 또는 null",
  "event_month": 1~12 숫자 또는 null,
  "budget_max": 숫자 또는 null,
  "quantity": 숫자 또는 null,
  "style_keywords": ["실용적", "고급", "저렴한"] 형태,
  "must_have": ["로고인쇄", "휴대성"] 형태
}}
"""

    try:
        response = ollama.chat(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt.strip()},
                {"role": "user", "content": user_prompt.strip()},
            ],
            options={"temperature": 0.0, "num_ctx": 2048},
            stream=False,
        )
        parsed = safe_json_loads(response["message"]["content"])

        result = fallback.copy()
        for k, v in parsed.items():
            if k in result:
                result[k] = v

        for k in ["budget_max", "quantity", "event_month", "season"]:
            if result.get(k) in [None, "", []] and fallback.get(k) not in [None, "", []]:
                result[k] = fallback[k]

        result["raw_query"] = query
        result["extraction_method"] = "ollama_llm"
        return result

    except Exception as e:
        fallback["extraction_error"] = str(e)
        return fallback


# 테스트
for q in [
    "8월 행사에서 나눠줄 여름 판촉물 추천해줘",
    "병원 개원 답례품으로 3천원 이하 500개 추천해줘",
]:
    print("\nQUERY:", q)
    print(extract_query_condition(q))

## 8. 행사/사용월 기준 참고 주문월 계산

*v1과 동일한 로직입니다.*

In [ ]:
def prev_month(month: int, n: int = 1) -> int:
    m = month - n
    while m <= 0:
        m += 12
    return m


def get_month_weights(event_month: int) -> dict:
    if event_month is None or pd.isna(event_month):
        return {}
    event_month = int(event_month)
    return {
        prev_month(event_month, 1): 1.00,
        prev_month(event_month, 2): 0.85,
        event_month: 0.45,
        prev_month(event_month, 3): 0.25,
    }


def get_event_months(query_condition: dict) -> list:
    months = []
    event_month = query_condition.get("event_month")
    if event_month not in [None, "", np.nan]:
        try:
            months.append(int(event_month))
        except Exception:
            pass

    season = query_condition.get("season")
    if season in SEASON_MONTHS:
        months.extend(SEASON_MONTHS[season])

    return sorted(set(months))


def get_reference_month_weights(query_condition: dict) -> dict:
    combined = {}
    for m in get_event_months(query_condition):
        for month, score in get_month_weights(m).items():
            combined[month] = max(combined.get(month, 0), score)
    return combined


def calc_date_lead_score(order_month, query_condition: dict) -> float:
    if pd.isna(order_month):
        return 0.0
    weights = get_reference_month_weights(query_condition)
    if not weights:
        return 0.0
    try:
        return float(weights.get(int(order_month), 0.0))
    except Exception:
        return 0.0


# 테스트
for q in ["8월 행사 판촉물", "여름 행사 판촉물", "12월 연말 선물"]:
    cond = extract_basic_conditions(q)
    print(q, "->", get_reference_month_weights(cond))

## 9. 임베딩 생성 (상품 row + 거래 Profile)

*RAG 단계: Retrieval 준비 — v1과 다른 점은 거래 쪽 임베딩 대상이 row가 아니라 6번 섹션에서 만든
`trade_profile_df`라는 점입니다. 상품은 v1과 동일하게 row 단위입니다.*

In [ ]:
def try_ollama_embed_texts(texts, model=EMBED_MODEL, batch_size=32):
    """Ollama Python 패키지의 embed()/embeddings() API 차이를 모두 지원합니다."""
    import ollama

    all_embeddings = []
    for start in range(0, len(texts), batch_size):
        batch = [str(x) for x in texts[start:start + batch_size]]
        try:
            resp = ollama.embed(model=model, input=batch)
            if "embeddings" in resp:
                all_embeddings.extend(resp["embeddings"])
                continue
        except Exception:
            pass

        for text in batch:
            try:
                resp = ollama.embeddings(model=model, prompt=text)
                all_embeddings.append(resp["embedding"])
            except Exception as e:
                raise RuntimeError(f"Ollama embedding 실패: {e}")

    return np.array(all_embeddings, dtype=np.float32)


def build_search_index(name, texts, use_ollama=True, model=EMBED_MODEL):
    """임베딩 인덱스를 만듭니다. 캐시가 있으면 재사용합니다."""
    texts = [str(x) for x in texts]
    cache_path = CACHE_DIR / f"{name}_{model}_{text_hash(texts)}.npy"

    if use_ollama:
        try:
            if cache_path.exists():
                matrix = normalize_vector_matrix(np.load(cache_path))
                print(f"[{name}] 임베딩 캐시 로딩: {cache_path.name}")
                return {"backend": "ollama", "model": model, "matrix": matrix, "vectorizer": None}

            print(f"[{name}] Ollama 임베딩 생성 중... rows={len(texts)}")
            matrix = try_ollama_embed_texts(texts, model=model)
            np.save(cache_path, matrix)
            return {"backend": "ollama", "model": model, "matrix": normalize_vector_matrix(matrix), "vectorizer": None}

        except Exception as e:
            warnings.warn(f"[{name}] Ollama 임베딩 실패, TF-IDF로 대체합니다: {e}")

    print(f"[{name}] TF-IDF fallback 생성")
    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=1)
    matrix = vectorizer.fit_transform(texts)
    return {"backend": "tfidf", "model": "tfidf_char_ngram", "matrix": matrix, "vectorizer": vectorizer}


def search_index(query_text, index) -> np.ndarray:
    """인덱스 backend에 맞춰 query와 문서들 간 유사도 점수를 계산합니다."""
    if index["backend"] == "ollama":
        q_vec = normalize_vector_matrix(try_ollama_embed_texts([query_text], model=index["model"]))
        return cosine_scores(q_vec[0], index["matrix"])
    if index["backend"] == "tfidf":
        q_vec = index["vectorizer"].transform([query_text])
        return cosine_similarity(q_vec, index["matrix"]).flatten()
    raise ValueError(f"지원하지 않는 backend: {index['backend']}")


# 상품: row 단위 (v1과 동일)
product_index = build_search_index("product", product_df["product_search_text"], use_ollama=USE_OLLAMA_EMBEDDING)

# 거래: profile 단위 (v1과 다른 부분)
trade_profile_index = build_search_index("trade_profile", trade_profile_df["trade_profile_search_text"], use_ollama=USE_OLLAMA_EMBEDDING)

print("product backend:", product_index["backend"])
print("trade_profile backend:", trade_profile_index["backend"])
print("상품 row 수:", len(product_df))
print("거래 profile 수:", len(trade_profile_df))

## 10. 유사 거래 Profile 검색

*RAG 단계: Retrieval — v1의 `retrieve_similar_trades`에 대응합니다.
"거래량 점수"가 추가되어, 거래가 많이 쌓인 profile을 조금 더 신뢰합니다.*

```text
profile_total_score = 의미유사도(50%) + 날짜리드타임(20%) + 예산/수량/시즌 조건(15%) + 거래량 점수(15%)
```

In [ ]:
def build_query_text(query_condition: dict) -> str:
    """query_condition(구조화된 질문)을 검색 쿼리 문자열로 합칩니다."""
    parts = [
        query_condition.get("raw_query", ""),
        query_condition.get("buyer_context", ""),
        query_condition.get("event_context", ""),
        query_condition.get("purpose", ""),
        query_condition.get("season", ""),
        " ".join(query_condition.get("style_keywords") or []),
        " ".join(query_condition.get("must_have") or []),
    ]
    return combine_text(*parts)


def calc_profile_condition_score(row, query_condition: dict) -> float:
    """예산/수량/시즌이 이 거래 profile과 얼마나 맞는지 0~1 점수로 계산합니다."""
    score = 0.0

    budget = query_condition.get("budget_max")
    if budget not in [None, ""] and not pd.isna(budget):
        prices = [p for p in [row.get("median_bulk_price"), row.get("median_middle_price"), row.get("median_small_price")] if not pd.isna(p)]
        if prices:
            min_price = min(prices)
            if min_price <= float(budget):
                score += 0.35
            elif min_price <= float(budget) * 1.2:
                score += 0.15

    quantity = query_condition.get("quantity")
    if quantity not in [None, ""] and not pd.isna(quantity):
        moq = row.get("median_moq", np.nan)
        if pd.isna(moq) or moq <= float(quantity):
            score += 0.25

    season = query_condition.get("season")
    if season and season in str(row.get("season_filter", "")):
        score += 0.20

    return min(score, 1.0)


def retrieve_similar_trade_profiles(query: str, query_condition: dict, top_k: int = 20) -> pd.DataFrame:
    """질문과 가장 관련 있는 거래 profile top_k건을 점수와 함께 돌려줍니다."""
    query_text = build_query_text(query_condition)
    semantic_scores = search_index(query_text, trade_profile_index)

    result = trade_profile_df.copy()
    result["profile_semantic_score"] = semantic_scores
    result["date_lead_score"] = result["order_month"].apply(lambda m: calc_date_lead_score(m, query_condition))
    result["profile_condition_score"] = result.apply(lambda row: calc_profile_condition_score(row, query_condition), axis=1)

    result["profile_total_score"] = (
        result["profile_semantic_score"] * 0.50 +
        result["date_lead_score"] * 0.20 +
        result["profile_condition_score"] * 0.15 +
        result["trade_count_score"] * 0.15
    )

    return result.sort_values("profile_total_score", ascending=False).head(top_k).reset_index(drop=True)


# 테스트
test_query = "8월 행사에서 나눠줄 여름 판촉물 추천해줘"
test_condition = extract_query_condition(test_query)
similar_profiles = retrieve_similar_trade_profiles(test_query, test_condition, top_k=10)

print("[추출된 조건]")
print(json.dumps(test_condition, ensure_ascii=False, indent=2))

display(similar_profiles[[
    "trade_profile_id", "profile_total_score", "profile_semantic_score",
    "date_lead_score", "profile_condition_score", "trade_count_score", "trade_count",
    "order_month", "buyer_type_path", "trade_category_path", "sample_products",
    "event_filter", "target_filter", "season_filter",
]])

## 11. 거래 Profile 신호 추출

*RAG 단계: Retrieval 결과 재구성 — v1의 `extract_trade_signals`에 대응합니다.
profile은 여러 거래가 섞여 있으므로, 대표상품(`sample_products`)을 다시 낱개로 풀어서 집계하고,
유사도뿐 아니라 실제 거래량(`trade_count`)도 함께 반영합니다.*

In [ ]:
def weighted_value_counts_from_profiles(df, col, weight_col="profile_total_score", top_n=10):
    """유사도(weight)와 실제 거래량(trade_count)을 함께 반영한 가중 집계입니다."""
    rows = []
    for _, row in df.iterrows():
        value = clean_text(row.get(col, ""))
        if not value:
            continue
        rows.append((value, float(row.get(weight_col, 0)), int(row.get("trade_count", 1))))

    if not rows:
        return pd.DataFrame(columns=[col, "weighted_score", "profile_count", "trade_count_sum", "signal_score"])

    temp = pd.DataFrame(rows, columns=[col, "weight", "trade_count"])
    out = (
        temp.groupby(col)
        .agg(weighted_score=("weight", "sum"), profile_count=("weight", "count"), trade_count_sum=("trade_count", "sum"))
        .reset_index()
    )
    # 유사도 점수 * log(거래량) — 유사도만으로도, 거래량만으로도 한쪽에 치우치지 않게 함
    out["signal_score"] = out["weighted_score"] * np.log1p(out["trade_count_sum"])
    return out.sort_values("signal_score", ascending=False).head(top_n).reset_index(drop=True)


def extract_profile_signals(similar_profiles: pd.DataFrame) -> dict:
    """유사 거래 profile들을 '신호 텍스트' 하나로 요약합니다 (v1의 extract_trade_signals에 대응)."""
    signal_cols = {
        "category_large": ("trade_category_large", "유사거래 상품분류 대"),
        "category_middle": ("trade_category_middle", "유사거래 상품분류 중"),
        "category_small": ("trade_category_small", "유사거래 상품분류 소"),
        "buyers": ("buyer_type_middle", "유사구매처"),
        "events": ("event_filter", "행사"),
        "targets": ("target_filter", "대상"),
        "seasons": ("season_filter", "시즌"),
    }

    signals = {}
    signal_parts = []
    for key, (col, title) in signal_cols.items():
        counts = weighted_value_counts_from_profiles(similar_profiles, col, top_n=10)
        signals[key] = counts
        top_values = counts[col].head(8).tolist() if len(counts) else []
        if top_values:
            signal_parts.append(f"[{title}] " + " ".join(top_values))

    # 대표상품(sample_products)은 '/'로 묶여 있으므로 다시 낱개로 풀어서 집계
    product_rows = []
    for _, row in similar_profiles.iterrows():
        for item in clean_text(row.get("sample_products", "")).split("/"):
            item = clean_text(item)
            if item:
                product_rows.append({"sample_product": item, "weight": float(row.get("profile_total_score", 0)), "trade_count": int(row.get("trade_count", 1))})

    if product_rows:
        temp = pd.DataFrame(product_rows)
        products = (
            temp.groupby("sample_product")
            .agg(weighted_score=("weight", "sum"), count=("weight", "count"), trade_count_sum=("trade_count", "sum"))
            .reset_index()
        )
        products["signal_score"] = products["weighted_score"] * np.log1p(products["trade_count_sum"])
        products = products.sort_values("signal_score", ascending=False).head(10).reset_index(drop=True)
    else:
        products = pd.DataFrame(columns=["sample_product", "weighted_score", "count", "trade_count_sum", "signal_score"])

    signals["products"] = products
    if len(products):
        signal_parts.append("[유사거래 대표상품] " + " ".join(products["sample_product"].head(8).tolist()))

    signals["signal_text"] = clean_text("\n".join(signal_parts))
    return signals


signals = extract_profile_signals(similar_profiles)

print("[거래 Profile 신호 텍스트]")
print(signals["signal_text"])

print("\n[상위 상품분류 중]")
display(signals["category_middle"])

print("\n[상위 대표상품]")
display(signals["products"])

## 12. 상품 후보 랭킹

*RAG 단계: Retrieval 결과 활용 — v1의 `rank_products`와 구조는 같지만,
거래 Profile 신호가 row보다 더 정제된 정보이기 때문에 **거래신호 비중을 v1보다 높게** 둡니다.*

| 점수 | v1 비중 | v2 비중 |
|---|---:|---:|
| query_product_score | 20% | 18% |
| trade_signal_product_score | 30% | 32% |
| category_signal_score | 20% | 22% |
| budget_score | 15% | 15% |
| quantity_score | 5% | 5% |
| product_quality_score | 10% | 8% |

In [ ]:
def calc_budget_score(price, budget):
    if budget in [None, ""] or pd.isna(budget):
        return 0.5
    if pd.isna(price):
        return 0.2
    price, budget = float(price), float(budget)
    if price <= budget:
        return 1.0
    if price <= budget * 1.2:
        return 0.65
    if price <= budget * 1.5:
        return 0.35
    return 0.0


def calc_quantity_score(moq, quantity):
    if quantity in [None, ""] or pd.isna(quantity):
        return 0.5
    if pd.isna(moq):
        return 0.7
    moq, quantity = float(moq), float(quantity)
    if moq <= quantity:
        return 1.0
    if moq <= quantity * 1.5:
        return 0.5
    return 0.0


def calc_product_quality_score(row):
    score = 0.0
    if clean_text(row.get("product_name", "")):
        score += 0.25
    if clean_text(row.get("category_path", "")):
        score += 0.25
    if not pd.isna(row.get("price", np.nan)):
        score += 0.25
    if clean_text(row.get("keywords", "")) or clean_text(row.get("summary", "")):
        score += 0.25
    return score


def calc_profile_category_signal_score(row, signals: dict) -> float:
    """거래 Profile에서 추출된 상품분류 신호가 현재 상품 텍스트에 얼마나 겹치는지 봅니다."""
    product_text = combine_text(row.get("category_path", ""), row.get("product_name", ""), row.get("keywords", ""))

    score = 0.0
    for signal_key, col_name, weight in [
        ("category_middle", "trade_category_middle", 0.55),
        ("category_small", "trade_category_small", 0.35),
        ("category_large", "trade_category_large", 0.10),
    ]:
        signal_df = signals.get(signal_key)
        if signal_df is None or len(signal_df) == 0:
            continue
        max_signal = signal_df["signal_score"].max()
        for _, srow in signal_df.iterrows():
            value = clean_text(srow.get(col_name, ""))
            if value and value in product_text:
                score += (float(srow["signal_score"]) / max_signal if max_signal else 0) * weight

    return min(score, 1.0)


def make_recommend_reason(row, budget, quantity) -> str:
    reasons = []
    if row["trade_signal_product_score"] >= 0.55:
        reasons.append("거래 Profile 신호와 의미적으로 가까움")
    elif row["trade_signal_product_score"] >= 0.35:
        reasons.append("거래 Profile 신호와 일부 관련 있음")

    if row["profile_category_signal_score"] >= 0.5:
        reasons.append("유사 거래 Profile의 상품분류와 현재 상품분류가 일치")
    elif row["profile_category_signal_score"] > 0:
        reasons.append("유사 거래 Profile의 상품분류와 일부 겹침")

    if row["query_product_score"] >= 0.55:
        reasons.append("질문과 상품 설명의 의미 유사도 높음")

    if budget not in [None, ""] and not pd.isna(budget) and pd.notna(row["price"]):
        reasons.append(f"예산 {int(float(budget)):,}원 이하 조건 충족" if float(row["price"]) <= float(budget) else "예산 초과 여부 확인 필요")

    if quantity not in [None, ""] and not pd.isna(quantity):
        moq_ok = pd.isna(row["moq"]) or float(row["moq"]) <= float(quantity)
        reasons.append("요청 수량 기준 최소구매수량 충족 가능" if moq_ok else "최소구매수량 확인 필요")

    return " / ".join(reasons) if reasons else "현재 상품데이터 기준 추천 후보로 검토 가능"


def rank_products_profile(query: str, query_condition: dict, signals: dict, top_k: int = 5) -> pd.DataFrame:
    query_text = build_query_text(query_condition)

    query_product_scores = search_index(query_text, product_index)
    signal_text = signals.get("signal_text", "")
    trade_signal_product_scores = search_index(signal_text, product_index) if signal_text else np.zeros(len(product_df))

    result = product_df.copy()
    result["query_product_score"] = query_product_scores
    result["trade_signal_product_score"] = trade_signal_product_scores
    result["profile_category_signal_score"] = result.apply(lambda row: calc_profile_category_signal_score(row, signals), axis=1)

    budget = query_condition.get("budget_max")
    quantity = query_condition.get("quantity")
    result["budget_score"] = result["price"].apply(lambda p: calc_budget_score(p, budget))
    result["quantity_score"] = result["moq"].apply(lambda q: calc_quantity_score(q, quantity))
    result["product_quality_score"] = result.apply(calc_product_quality_score, axis=1)

    # v1보다 거래신호(Profile) 비중을 높게 둠
    result["final_score"] = (
        result["query_product_score"] * 0.18 +
        result["trade_signal_product_score"] * 0.32 +
        result["profile_category_signal_score"] * 0.22 +
        result["budget_score"] * 0.15 +
        result["quantity_score"] * 0.05 +
        result["product_quality_score"] * 0.08
    )

    result["recommend_reason"] = result.apply(lambda row: make_recommend_reason(row, budget, quantity), axis=1)

    output_cols = [
        "product_id", "product_name", "price", "moq", "category_path", "keywords",
        "query_product_score", "trade_signal_product_score", "profile_category_signal_score",
        "budget_score", "quantity_score", "product_quality_score",
        "final_score", "recommend_reason", "image",
    ]
    return result.sort_values("final_score", ascending=False).head(top_k)[output_cols].reset_index(drop=True)

## 13. 최종 추천 함수

*RAG 단계 전체 통합 — v1의 `recommend_products_v1()`에 대응합니다.*

```text
질문 → 조건 추출 → 참고월 계산 → 유사 거래 Profile 검색 → Profile 신호 추출 → 상품 랭킹
```

In [ ]:
def recommend_products_v2(query: str, top_k: int = DEFAULT_TOP_K, profile_top_k: int = 30) -> dict:
    query_condition = extract_query_condition(query)
    similar_profiles = retrieve_similar_trade_profiles(query, query_condition, top_k=profile_top_k)
    signals = extract_profile_signals(similar_profiles)
    recommendations = rank_products_profile(query, query_condition, signals, top_k=top_k)

    return {
        "query": query,
        "query_condition": query_condition,
        "reference_month_weights": get_reference_month_weights(query_condition),
        "similar_profiles": similar_profiles,
        "signals": signals,
        "recommendations": recommendations,
        "backend": {
            "product": product_index["backend"],
            "trade_profile": trade_profile_index["backend"],
            "embedding_model": product_index.get("model"),
        },
        "counts": {
            "trade_rows": len(trade_df),
            "trade_profiles": len(trade_profile_df),
            "products": len(product_df),
        },
    }


# 테스트
test_query = "8월 행사에서 나눠줄 여름 판촉물 추천해줘"
result = recommend_products_v2(test_query, top_k=5, profile_top_k=20)

print("[질문]", result["query"])
print("\n[추출된 조건]")
print(json.dumps(result["query_condition"], ensure_ascii=False, indent=2))
print("\n[사용 backend]", result["backend"])
print("\n[데이터 수]", result["counts"])

print("\n[추천 상품]")
display(result["recommendations"])

## 14. 여러 질문 일괄 테스트 및 결과 저장

In [ ]:
test_queries = [
    "8월 행사에서 나눠줄 여름 판촉물 추천해줘",
    "병원 개원 답례품으로 3천원 이하 500개 추천해줘",
    "대학교 OT에서 나눠줄 저렴한 사은품 추천해줘",
    "회사 창립기념품으로 실용적인 상품 추천해줘",
    "박람회 부스에서 나눠줄 홍보물 추천해줘",
    "12월 연말 고객 선물로 5만원 이하 상품 추천해줘",
    "어린이집 행사 답례품 추천해줘",
    "로고 인쇄 가능한 텀블러 추천해줘",
]

all_rows = []
for query in test_queries:
    r = recommend_products_v2(query, top_k=5, profile_top_k=20)
    cond = r["query_condition"]

    for rank, (_, row) in enumerate(r["recommendations"].iterrows(), start=1):
        all_rows.append({
            "query": query,
            "rank": rank,
            "buyer_context": cond.get("buyer_context", ""),
            "event_context": cond.get("event_context", ""),
            "purpose": cond.get("purpose", ""),
            "season": cond.get("season", ""),
            "event_month": cond.get("event_month", ""),
            "product_id": row["product_id"],
            "product_name": row["product_name"],
            "price": row["price"],
            "moq": row["moq"],
            "category_path": row["category_path"],
            "final_score": row["final_score"],
            "recommend_reason": row["recommend_reason"],
            "trade_rows": r["counts"]["trade_rows"],
            "trade_profiles": r["counts"]["trade_profiles"],
        })

batch_result_df = pd.DataFrame(all_rows)
batch_result_path = OUTPUT_DIR / "giftco_product_recommendation_v2_profile_260723_result.xlsx"
batch_result_df.to_excel(batch_result_path, index=False)

print("결과 저장:", batch_result_path)
display(batch_result_df.head(20))

## 15. Ollama로 고객용 추천 답변 생성

*v1과 동일한 구조입니다. 컨텍스트에 들어가는 근거가 거래 row 대신 거래 Profile이라는 점만 다릅니다.*

In [ ]:
def build_answer_context(result: dict) -> str:
    """LLM에게 넘길 컨텍스트를 하나의 텍스트로 구성합니다."""
    lines = ["[질문]", result["query"], ""]
    lines += ["[추출된 조건]", json.dumps(result["query_condition"], ensure_ascii=False), ""]
    lines += ["[참고 주문월 가중치]", json.dumps(result["reference_month_weights"], ensure_ascii=False), ""]
    lines += ["[거래 Profile 신호]", result["signals"].get("signal_text", ""), ""]

    lines.append("[추천 상품 후보]")
    for i, (_, row) in enumerate(result["recommendations"].iterrows(), start=1):
        lines += [
            f"{i}. {row['product_name']}",
            f"   - 가격: {row['price']}",
            f"   - 최소구매수량: {row['moq']}",
            f"   - 카테고리: {row['category_path']}",
            f"   - 추천근거: {row['recommend_reason']}",
            "",
        ]

    return "\n".join(lines)


def generate_customer_answer(query: str, model: str = LLM_MODEL, top_k: int = 5) -> dict:
    import ollama

    result = recommend_products_v2(query, top_k=top_k, profile_top_k=30)
    context = build_answer_context(result)

    system_prompt = """
너는 기프트코 판촉물 상품 추천 어시스턴트다.

규칙:
- 제공된 추천 후보와 거래 Profile 신호만 근거로 답변한다.
- 데이터에 없는 납기, 재고, 인쇄 가능 여부는 단정하지 않는다.
- 가격과 최소구매수량은 제공된 값만 말한다.
- 고객에게 보여줄 수 있게 자연스러운 한국어로 답변한다.
- 상위 3~5개 상품을 추천하고, 각 상품의 추천 이유를 짧게 적는다.
- 마지막에 확인이 필요한 조건을 적는다.
"""

    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt.strip()},
            {"role": "user", "content": f"아래 근거를 바탕으로 고객에게 상품 추천 답변을 작성해줘.\n\n{context}"},
        ],
        options={"temperature": 0.2, "num_ctx": 4096},
        stream=False,
    )

    return {"answer": response["message"]["content"], "result": result}


# 사용 예시
answer_result = generate_customer_answer("8월 행사에서 나눠줄 여름 판촉물 추천해줘", top_k=5)
print(answer_result["answer"])
display(answer_result["result"]["recommendations"])

# 다음 단계

이 v2(profile 단위)를 실행한 뒤 확인할 것:

1. 거래 profile 수가 row 수보다 충분히 줄었는가 (감소율 확인)?
2. profile 집계 때문에 너무 뭉뚱그려져서 구체적인 상품 신호가 사라지지는 않았는가?
3. 거래량 점수(`trade_count_score`)가 결과를 한쪽으로 과도하게 쏠리게 만들지는 않는가?
4. v1(row)과 v2(profile)의 추천 결과를 같은 질문으로 비교했을 때 차이가 합리적인가?

## 다음 버전

다음: v3에서는 현재 상품 검색을 추천의 중심에 두고, 거래 Profile은 순위 보정 신호로만 활용합니다 (가격/MOQ/판매상태 필터, 관련도 임계값, 평가셋 도입 포함). 전체 계획은 `README.md`의 "13. 다음 단계: v3 계획"을 참고하세요.
